OCR

In [2]:
import easyocr
import cv2
import numpy as np
from pdf2image import convert_from_path
import os
from typing import List, Dict

In [3]:
class OCRAgent:
    def __init__(self, languages: List[str] = ["en"]):
        self.reader = easyocr.Reader(languages)

    def preprocess(self, image_path: str):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.medianBlur(image, 3)
        _, thresh = cv2.threshold(image, 150, 255, cv2.THRESH_BINARY)
        return thresh

    def ocr_image(self, image_path: str) -> Dict:
        processed = self.preprocess(image_path)
        results = self.reader.readtext(processed)

        text_blocks = []
        confidences = []

        for bbox, text, confidence in results:
            text_blocks.append(text)
            confidences.append(confidence)

        return {
            "raw_text": "\n".join(text_blocks),
            "avg_confidence": np.mean(confidences) if confidences else 0.0
        }

    def ocr_pdf(self, pdf_path: str) -> List[Dict]:
        pages = convert_from_path(pdf_path)
        output = []

        for i, page in enumerate(pages):
            temp_path = f"temp_page_{i}.png"
            page.save(temp_path, "PNG")

            page_result = self.ocr_image(temp_path)
            page_result["page_number"] = i + 1
            output.append(page_result)

            os.remove(temp_path)

        return output

In [4]:
ocr_agent = OCRAgent()

result = ocr_agent.ocr_image("test-samples/test2.png")
print(result["avg_confidence"])
print(result["raw_text"])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\tosha\Desktop\Projects\medical-ingestion-system\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


0.8778257082951045
AFs
Hemolab
Blood Test Results
Johnatan Doe
Processing Details
Central Health Laboratory
Age:
30
Sample:
2024-01-20 08.30 AM
123 Health Ave; Medicity
Gender:
Male
Results:
2024-01-22 03.00 PM
12345 Wellbeing;
Email:
johndoe@example.com
Verified by:
Dr: Emily Johnson
Healthland
Hematology
TEST
UNiT
REFERENCE RANGE
RESULT
RESULT STATUS
White Blood Cell Count
X10^9/L
4 - 11
7.2
Normal
Red Blood Cell Count
X10^12/L
4.2-5.9
4.5
Normal
Hemoglobin
g/dL
13.2-16.6
13.5
Normal
Hematocrit
%
38.3
48.6
40.1
Normal
Platelet Count
X10^9/L
150
450
250
Normal
Biochemistry
TEST
UNit
REFERENCE RANGE
RESULT
RESULT STATUS
Glucose
mg/dl
70-
99
90
Normal
Creatinine
mg/dL
0.6 -1,.2
0.9
Normal
Urea
mg/dL
15 - 50
29
Normal
Cholesterol
mg/dL
0 - 200
185
Normal
Alanine Aminotransferase (ALT)
U/L
7
56
25
Normal
Liver Function
TEST
UNiT
REFERENCE RANGE
RESULT
RESULT STATUS
Aspartate Aminotransferase (AST)
U/L
8 - 48
30
Normal
Alkaline Phosphatase (ALP)
U/L
40
129
70
Normal
Bilirubin
mg/dL
0.3-1.9

In [5]:
pdf_result = ocr_agent.ocr_pdf("test-samples/test1.pdf")

for page in pdf_result:
    print(f"\n--- Page {page['page_number']} ---")
    print("Confidence:", page["avg_confidence"])
    print(page["raw_text"])


--- Page 1 ---
Confidence: 0.8869123908077813
Hemolab
Blood Test Result
Central Health Laboratory
123 Health Ave, Medicity
Johnatan Doe
Processing Details
12345 Wellbeing
Age:
30
Sample:
7/23/2024, 12.51 PM
Healthland
Gender:
Male
Results:
7/23/2024, 12.51 PM
Email:
johndoe@example.com
Verified by: Dr. Emily Johnson
Hematology
TEST
RESULT
UNiT
NORMAL RANGE
RESULT STATUS
White Blood Cell Count
7.2
X10^9/L
4
11
Normal
Red Blood Cell Count
14.5
X10^1Z/L
4.2
5.9
Elevated
Hemoglobin
13.5
13.2
16.6
Normal
Hematocrit
40.1
%6
38.3
48.6
Normal
Platelet Count
250
X10^9/L
150
450
Normal
Biochemistry
TEST
RESULT
UNit
NORMAL RANGE
RESULT STATUS
Glucose
90
mgdL
70
99
Normal
Creatinine
9
mgdL
0.6
1.2
Normal
Urea
29
mgdL
15
50
Normal
Cholesterol
185
mgdl
0
200
Normal
Alanine Aminotransferase (ALT)
25
UIL
56
Normal
Liver function
TEST
RESULT
UNiT
NORMAL RANGE
RESULT STATUS
Aspartate Aminotransferase
30
UIL
8
48
Normal
(AST)
Alkaline Phosphatase (ALP)
70
UIL
40
129
Normal
Bilirubin
1.2
mgdL
0.3 -
Norma

Pre-Processing

In [6]:
import re

class CleaningAgent:
    def clean_text(self, text: str) -> str:
        # Remove multiple spaces
        text = re.sub(r'\s+', ' ', text)

        # Remove weird OCR artifacts (common)
        text = re.sub(r'[|]', '', text)

        # Fix common OCR mistakes (expand later)
        replacements = {
            "0": "O",  # example (careful with numbers)
        }

        for wrong, correct in replacements.items():
            text = text.replace(wrong, correct)

        return text.strip()
    

cleaner = CleaningAgent()
cleaned_text = cleaner.clean_text(result["raw_text"])
print(cleaned_text)

AFs Hemolab Blood Test Results Johnatan Doe Processing Details Central Health Laboratory Age: 3O Sample: 2O24-O1-2O O8.3O AM 123 Health Ave; Medicity Gender: Male Results: 2O24-O1-22 O3.OO PM 12345 Wellbeing; Email: johndoe@example.com Verified by: Dr: Emily Johnson Healthland Hematology TEST UNiT REFERENCE RANGE RESULT RESULT STATUS White Blood Cell Count X1O^9/L 4 - 11 7.2 Normal Red Blood Cell Count X1O^12/L 4.2-5.9 4.5 Normal Hemoglobin g/dL 13.2-16.6 13.5 Normal Hematocrit % 38.3 48.6 4O.1 Normal Platelet Count X1O^9/L 15O 45O 25O Normal Biochemistry TEST UNit REFERENCE RANGE RESULT RESULT STATUS Glucose mg/dl 7O- 99 9O Normal Creatinine mg/dL O.6 -1,.2 O.9 Normal Urea mg/dL 15 - 5O 29 Normal Cholesterol mg/dL O - 2OO 185 Normal Alanine Aminotransferase (ALT) U/L 7 56 25 Normal Liver Function TEST UNiT REFERENCE RANGE RESULT RESULT STATUS Aspartate Aminotransferase (AST) U/L 8 - 48 3O Normal Alkaline Phosphatase (ALP) U/L 4O 129 7O Normal Bilirubin mg/dL O.3-1.9 1.2 Normal Albumin

LLM

In [17]:
import os
import re
import json
from dotenv import load_dotenv
import google.generativeai as genai

# Load .env
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")


def safe_json_parse(text: str) -> dict:
    """
    Cleans Gemini output and extracts first valid JSON object.
    Handles markdown code blocks automatically.
    """
    text = text.strip()

    # Remove markdown code fences if present
    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    # Extract first JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        print("Raw model output:")
        print(text)
        raise ValueError("No JSON object found in response")

    return json.loads(match.group())


def extract_medical_data(ocr_text: str) -> dict:
    prompt = f"""
You are a medical data extraction system.

STRICT RULES:
- Extract ONLY information explicitly present in the OCR text.
- Do NOT infer or invent lab tests.
- Do NOT add missing medical values.
- If a field is not present, return null or empty list.
- Return ONLY raw JSON. No markdown. No explanations.

Schema:
{{
  "patient_name": string | null,
  "age": string | null,
  "gender": string | null,
  "date_of_visit": string | null,
  "diagnosis": [string],
  "medications": [
      {{
        "name": string,
        "dosage": string | null,
        "frequency": string | null
      }}
  ],
  "lab_results": [
      {{
        "test_name": string,
        "value": string,
        "unit": string | null,
        "reference_range": string | null
      }}
  ]
}}

OCR TEXT:
\"\"\"
{ocr_text}
\"\"\"
"""

    response = model.generate_content(prompt)
    return safe_json_parse(response.text)

In [18]:
cleaned_text = cleaner.clean_text(result["raw_text"])
structured_data = extract_medical_data(cleaned_text)

structured_data

{'patient_name': 'Johnatan Doe',
 'age': '3O',
 'gender': 'Male',
 'date_of_visit': None,
 'diagnosis': [],
 'medications': [],
 'lab_results': [{'test_name': 'White Blood Cell Count',
   'value': '7.2',
   'unit': 'X1O^9/L',
   'reference_range': '4 - 11'},
  {'test_name': 'Red Blood Cell Count',
   'value': '4.5',
   'unit': 'X1O^12/L',
   'reference_range': '4.2-5.9'},
  {'test_name': 'Hemoglobin',
   'value': '13.5',
   'unit': 'g/dL',
   'reference_range': '13.2-16.6'},
  {'test_name': 'Hematocrit',
   'value': '4O.1',
   'unit': '%',
   'reference_range': '38.3 48.6'},
  {'test_name': 'Platelet Count',
   'value': '25O',
   'unit': 'X1O^9/L',
   'reference_range': '15O 45O'},
  {'test_name': 'Glucose',
   'value': '9O',
   'unit': 'mg/dl',
   'reference_range': '7O- 99'},
  {'test_name': 'Creatinine',
   'value': 'O.9',
   'unit': 'mg/dL',
   'reference_range': 'O.6 -1,.2'},
  {'test_name': 'Urea',
   'value': '29',
   'unit': 'mg/dL',
   'reference_range': '15 - 5O'},
  {'test_n

Structured Saving

In [19]:
import pandas as pd

def export_to_csv(structured_data, filename="medical_output.csv"):
    # Patient-level info
    patient_info = {
        "patient_name": structured_data.get("patient_name"),
        "age": structured_data.get("age"),
        "gender": structured_data.get("gender"),
        "date_of_visit": structured_data.get("date_of_visit"),
    }

    # Lab results table
    lab_results = structured_data.get("lab_results", [])

    df = pd.DataFrame(lab_results)

    # Save CSV
    df.to_csv(filename, index=False)

    print(f"Lab results saved to {filename}")

    # Also print patient info separately
    print("\nPATIENT INFO")
    for k, v in patient_info.items():
        print(f"{k}: {v}")

export_to_csv(structured_data)

Lab results saved to medical_output.csv

PATIENT INFO
patient_name: Johnatan Doe
age: 3O
gender: Male
date_of_visit: None


In [20]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import fonts
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from reportlab.lib.styles import getSampleStyleSheet

def export_to_pdf(structured_data, filename="medical_report.pdf"):
    doc = SimpleDocTemplate(filename)
    elements = []
    styles = getSampleStyleSheet()

    # Title
    elements.append(Paragraph("<b>Medical Report Summary</b>", styles["Title"]))
    elements.append(Spacer(1, 0.3 * inch))

    # Patient Info
    patient_info = [
        ["Patient Name", structured_data.get("patient_name")],
        ["Age", structured_data.get("age")],
        ["Gender", structured_data.get("gender")],
        ["Date of Visit", structured_data.get("date_of_visit")]
    ]

    patient_table = Table(patient_info, colWidths=[2.5 * inch, 3 * inch])
    patient_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 1, colors.grey),
    ]))

    elements.append(patient_table)
    elements.append(Spacer(1, 0.5 * inch))

    # Lab Results
    lab_results = structured_data.get("lab_results", [])
    table_data = [["Test Name", "Value", "Unit", "Reference Range"]]

    for lab in lab_results:
        table_data.append([
            lab.get("test_name"),
            lab.get("value"),
            lab.get("unit"),
            lab.get("reference_range")
        ])

    lab_table = Table(table_data, repeatRows=1)
    lab_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 1, colors.grey),
    ]))

    elements.append(lab_table)

    doc.build(elements)
    print(f"PDF saved to {filename}")

export_to_pdf(structured_data)

PDF saved to medical_report.pdf
